# Endogenous actions: ACTOR vs OBSERVER — does *acting* (not just observing) reshape the latent?

**Direction:** `endogenous-action-interactive-world` · GRU · updated 2026-07-29. The follow-up left open by
`findings/object-individuation.md` (which was scoped to *exogenous* actions). Here the **actor** generates actions
from its own hidden state and *acts on the world it must then predict* (closed sensorimotor loop); the **observer**
is an identical network *fed the same actions* but which never acts.

Companion notebooks: `endogenous_grabbability.ipynb` (can the latent be *edited*?) and
`endogenous_agent_animations.ipynb` (what the agents actually do). Canonical run registry:
`ENDOGENOUS_RUNS.md`.

*Source:* `runs/endogenous/*` (`scripts/train_endogenous.py`) → `scripts/eval_endogenous.py` →
`runs/endogenous/eval_metrics.json`.

## Run definitions (copied from `ENDOGENOUS_RUNS.md` — this notebook stands alone)

**Roles.** *actor* = its policy head emits the action applied to the world; trained on next-step prediction **plus**
(at level 3) REINFORCE + value baseline flowing into the **shared GRU trunk**. *observer* = identical architecture,
sees the **same** observations, is **fed the actor's actions**, trained on prediction **only**, never acts.

**Levels.** *L1* = `shift` dynamics (action = position delta, frustum/collision-guarded, so **death is impossible**),
prediction only. *L2* = `force` dynamics (action = force → momentum), lethal (collision + wall), still **no goal**.
*L3* = `force`, lethal, **plus a survival goal** trained by REINFORCE.

**Suffix key:** no suffix / `b` = the original ("**weak**") configuration, differing only in seed; `s` = the
"**strong**" configuration; the trailing digit is the seed.

| code | descriptive label (used in figures) | level | goal | architecture | training | seed | purpose |
|---|---|---|---|---|---|---|---|
| `L1` | L1 shift · pred-only · 256h | 1 | none | 256 hidden, Linear enc/dec | 2 500 it | 0 | efference-copy ablation |
| `L2` | L2 force · pred-only · 256h | 2 | none | 256 hidden, Linear enc/dec | 2 500 it | 0 | physical dynamics, no goal |
| `L3` | L3 force+goal · weak · 256h · s0 | 3 | **survive** | 256 hidden, Linear enc/dec | 6 000 it | 0 | first-pass goal-directed actor |
| `L3b` | L3 force+goal · weak · 256h · s1 | 3 | survive | 256 hidden, Linear enc/dec | 6 000 it | **1** | seed replication of `L3` |
| `L2s0` | L2 force · pred-only · **strong** | 2 | none | **512 hidden, 2-layer MLP enc + residual MLP dec** | 12 000 it, 5-step free-run loss | 0 | **no-goal control at strong capacity** |
| `L3s0` | L3 force+goal · **strong** · s0 | 3 | survive | strong (as above) | 25 000 it, 5-step free-run loss | 0 | main strong actor |
| `L3s1` | L3 force+goal · **strong** · s1 | 3 | survive | strong | 25 000 it | **1** | seed replication of `L3s0` |

**"weak" vs "strong" means exactly two configurations:** weak = 256 hidden, single Linear encoder + decoder, no
multistep loss, 6 000 iterations. strong = 512 hidden, 2-layer MLP encoder + residual MLP decoder, a 5-step
free-run (multistep) loss, 25 000 iterations — introduced to test whether results were artifacts of a weak predictor.

## Metric definitions (formulas, units, better-direction)

| metric | formula | units | better |
|---|---|---|---|
| **position / velocity R² (linear)** | `1 − ‖Y−(Wh+b)‖²/‖Y−Ȳ‖²`, least-squares on the passive latent `h`, **held-out 30%** | — | ↑ |
| **position / velocity R² (MLP)** | same with a 2-layer MLP probe | — | ↑ |
| **fiber residual (linear / MLP)** | `mean ‖h − g(pos,vel)‖ / ‖h‖` | fraction of ‖h‖ | ↓ (0 = fully canonical) |
| **next-step RMSE** | `RMSE(decode([h, a]), obs_{t+1})` on clean frames | obs intensity [0,1] | ↓ |
| **mean reward per step** | mean of (+0.1 survive / −1.0 death) over the 64×48 collection window | reward/step | ↑ (**+0.1 is the maximum**) |
| **deaths per 1000 frames** | `1000 · deaths / (batch·rollout)` | deaths/1000 frames | ↓ (0 = never dies) |
| **survival (frames per life)** | `batch·rollout / max(deaths,1)` = `3072 / max(deaths,1)` | frames | ↑ — **see the caveat below** |

> **⚠ Two reporting caveats you must read before interpreting the training curves.**
> 1. **Why 3072 is a *measurement* cap.** Training is on-policy: each iteration collects a **fixed budget of
>    environment frames** — `batch = 64` parallel worlds × `rollout = 48` steps = **3072 frames** — and then does one
>    gradient update. `survival` is estimated *inside that budget only*, as `frames ÷ deaths observed`. If **zero**
>    deaths occur anywhere in those 3072 frames, the estimator cannot tell "lives 3072 frames" from "lives forever";
>    it just returns 3072. This is **right-censoring**: any lifetime longer than the observation window is truncated
>    to the window. It also makes the statistic *quantized* (3072, 1536, 1024, … for 1, 2, 3 deaths). **The world
>    itself has no frame limit** — an episode ends only on death. **`deaths per 1000 frames` is a rate, so it is
>    unbiased and unbounded; prefer it.** (An earlier version of this notebook plotted that rate with an inverted
>    formula, which made improvement look like deterioration — fixed 2026-07-29.)
> 2. **`mean reward` is per step, not per episode.** +0.1 for each surviving step and −1.0 on death, so a value near
>    **+0.1 means almost every step survived** (it is the ceiling), *not* that the agent survived only a few frames.
>    With γ=0.99 the value of continued survival is ≈ 0.1/(1−0.99) = **10**, so a death costs roughly −1 −10 = −11 in
>    return terms: the −1 death penalty is small per se, but losing the survival stream is what dominates.

In [ ]:
# [1] Setup — load eval metrics + training logs (light academic theme for all metric panels).
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
OK = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','grey':'#8a8f98','red':'#D55E00','sky':'#56B4E9'}
def style_ax(ax):
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25, lw=0.6)
M = json.load(open('../../../../runs/endogenous/eval_metrics.json'))
ORDER = ['L1','L2','L3','L3b','L2s0','L3s0','L3s1']
ALL_RUNS = [k for k in ORDER if k in M]
# MAIN comparison kept deliberately small and readable: the goal-directed actor at both
# configurations plus the strong no-goal control. L1/L2-weak are footnotes (full table below).
LEVELS = [k for k in ['L3','L3s0','L2s0'] if k in M]
# descriptive labels — never show a bare run code in a figure
LEVEL_LABEL = {
    'L1':   'L1 shift\nno goal',
    'L2':   'L2 force\nno goal',
    'L3':   'L3 goal\nweak',
    'L3b':  'L3 goal\nweak · seed 1',
    'L2s0': 'L2 no-goal\nstrong (control)',
    'L3s0': 'L3 goal\nstrong',
    'L3s1': 'L3 goal\nstrong · seed 1',
}
BATCH_ROLLOUT = 64 * 48   # frames collected per training iteration = the survival cap
print('runs loaded:', LEVELS)

In [ ]:
# [2] Fig 1 — the level-3 actor learns to survive. ONE seed each (weak vs strong).
#     Panel (a) is the unbounded death RATE computed from the raw per-iteration death counts;
#     panel (b) the per-step reward (ceiling +0.1); panel (c) the censored survival estimate.
SHOW = [r for r in ['L3', 'L3s0'] if r in M]        # weak seed 0, strong seed 0
COLR = {'L3': OK['orange'], 'L3s0': OK['blue']}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for k in SHOW:
    it, sv = zip(*M[k]['survival_curve']); _, rw = zip(*M[k]['reward_curve'])
    deaths = M[k]['deaths_curve']
    rate = [1000.0 * d / BATCH_ROLLOUT for d in deaths]   # deaths per 1000 frames
    lab = LEVEL_LABEL[k].replace('\n', ' ')
    axes[0].plot(it, rate, lw=1.7, label=lab, color=COLR[k])
    axes[1].plot(it, rw, lw=1.7, label=lab, color=COLR[k])
    axes[2].plot(it, sv, lw=1.7, label=lab, color=COLR[k])
axes[0].set_title('(a) death rate  (↓ better, unbounded)', fontsize=10)
axes[0].set_ylabel('deaths per 1000 frames'); axes[0].set_yscale('symlog', linthresh=0.5)
axes[1].axhline(0.1, color='k', ls='--', lw=1, label='ceiling +0.1 (every step survived)')
axes[1].axhline(0, color=OK['grey'], lw=0.8, ls=':')
axes[1].set_title('(b) mean reward PER STEP', fontsize=10); axes[1].set_ylabel('reward / step')
axes[2].axhline(BATCH_ROLLOUT, color=OK['red'], ls='--', lw=1.2,
                label=f'{BATCH_ROLLOUT} = frames collected per iteration\n(right-censoring limit, not a world limit)')
axes[2].set_title('(c) survival estimate — CENSORED at the window', fontsize=10); axes[2].set_ylabel('frames per life')
for ax in axes:
    ax.set_xlabel('training iteration'); style_ax(ax); ax.legend(fontsize=7)
fig.suptitle('Fig 1 — the level-3 actor learns the survival goal (REINFORCE); weak vs strong, one seed each',
             y=1.03, fontsize=12)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [3] Fig 2 — Recoverability: read (pos, vel) off the passive latent — actor vs observer, per run.
def grouped(ax, metric, title, ylab):
    x = np.arange(len(LEVELS)); w = 0.38
    av = [M[k]['actor'][metric] for k in LEVELS]; ov = [M[k]['observer'][metric] for k in LEVELS]
    ax.bar(x - w/2, av, w, label='actor', color=OK['blue'])
    ax.bar(x + w/2, ov, w, label='observer', color=OK['orange'])
    ax.set_xticks(x); ax.set_xticklabels([LEVEL_LABEL[k] for k in LEVELS], fontsize=7)
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylab, fontsize=9); style_ax(ax); ax.legend(fontsize=8)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
grouped(axes[0], 'pos_r2_lin', '(a) position R² — LINEAR probe', 'R² (↑)')
grouped(axes[1], 'vel_r2_lin', '(b) velocity R² — LINEAR probe', 'R² (↑)')
fig.suptitle('Fig 2 — recoverability of the physical (pos, vel) from the passive latent', y=1.03, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [4] Fig 3 — Canonicality: fiber residual ‖h−g(pos,vel)‖/‖h‖ (lower = h is more a clean function of the state).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
grouped(axes[0], 'fiber_lin', '(a) fiber residual — LINEAR g', 'resid / ‖h‖ (↓)')
grouped(axes[1], 'fiber_mlp', '(b) fiber residual — MLP g', 'resid / ‖h‖ (↓)')
fig.suptitle('Fig 3 — canonicality: is the passive latent a function of (pos, vel)?', y=1.03, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [5] Fig 4 — Headline: actor − observer Δ per run (↑ = acting HELPS) + next-step prediction RMSE.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))
ax = axes[0]; x = np.arange(len(LEVELS)); w = 0.26
deltas = {'position R² lin': ('pos_r2_lin', 1), 'velocity R² lin': ('vel_r2_lin', 1),
          'canonicality (−Δ fiber MLP)': ('fiber_mlp', -1)}
for j,(lab,(mk,sgn)) in enumerate(deltas.items()):
    d = [sgn*(M[k]['actor'][mk]-M[k]['observer'][mk]) for k in LEVELS]
    ax.bar(x + (j-1)*w, d, w, label=lab, color=[OK['blue'],OK['green'],OK['red']][j])
ax.axhline(0, color='k', lw=0.8); ax.set_xticks(x); ax.set_xticklabels([LEVEL_LABEL[k] for k in LEVELS], fontsize=7)
ax.set_title('(a) actor − observer Δ  (positive = acting helps)', fontsize=10)
ax.set_ylabel('Δ (actor − observer)', fontsize=9); style_ax(ax); ax.legend(fontsize=8)
grouped(axes[1], 'nextstep_rmse', '(b) next-step prediction RMSE', 'RMSE (↓)')
fig.suptitle('Fig 4 — the actor-vs-observer contrast (Δ) and predictive quality', y=1.03, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [6] Full metrics table — EVERY run (held-out; actor / observer).
rows = ['| metric | ' + ' | '.join(LEVEL_LABEL[k].replace('\n',' ') + ' (actor / obs)' for k in ALL_RUNS) + ' |',
        '|---|' + '---|'*len(ALL_RUNS)]
def fmt(k, mk, p=3): return f"{M[k]['actor'][mk]:.{p}f} / {M[k]['observer'][mk]:.{p}f}"
for lab, mk in [('position R² linear ↑','pos_r2_lin'),('position R² MLP ↑','pos_r2_mlp'),
                ('velocity R² linear ↑','vel_r2_lin'),('velocity R² MLP ↑','vel_r2_mlp'),
                ('fiber resid linear ↓','fiber_lin'),('fiber resid MLP ↓','fiber_mlp'),
                ('next-step RMSE ↓','nextstep_rmse')]:
    rows.append('| '+lab+' | ' + ' | '.join(fmt(k, mk, 4 if 'RMSE' in lab else 3) for k in ALL_RUNS) + ' |')
rows.append('| deaths per 1000 frames (final) | ' + ' | '.join(
    (f"{1000.0*M[k]['deaths_curve'][-1]/BATCH_ROLLOUT:.2f}" if M[k].get('deaths_curve') else '—') for k in ALL_RUNS) + ' |')
display(Markdown('**Main figures above show only** ' + ', '.join(LEVEL_LABEL[k].replace(chr(10),' ') for k in LEVELS)
                 + '. The remaining runs (levels 1–2 without a goal, and the second seeds) are footnotes and appear only in this table.'))
display(Markdown('\n'.join(rows)))

## How the loss balances prediction against survival (and whether prediction is even needed)

The actor's objective is a **weighted sum**:

```
actor_loss = predictor_loss                      # next-step obs MSE (weight 1.0, fixed)
           + 1.0  * policy_loss                  # REINFORCE:  −(logπ · advantage)
           + 0.5  * value_loss                   # value-head regression onto the return
           + 0.01 * entropy_bonus                # exploration
```
(the observer trains on `predictor_loss` alone). **These weights were never swept** — they are the first values
tried, so the prediction-vs-control balance is currently an *arbitrary, unvalidated* hyperparameter. That is a real
limitation of the present result and the most obvious knob to sweep next.

**Is the prediction term even needed for the task?** Strictly, no: survival could be learned by REINFORCE alone from
the reward, with no observation-prediction objective at all. Prediction is here for two reasons — (i) it is the
*subject of the research question* (we are studying what the predictive latent looks like, and need the actor and
observer to share an objective so the comparison is fair), and (ii) it is the standard model-based-RL move: a
prediction loss is an auxiliary representation-shaping signal (this is exactly the RSSM/Dreamer pattern — the world
model is trained by reconstruction while the actor is trained on returns). **So the architecture deliberately mixes
two objectives that are not intrinsically coupled, and their relative weight is the thing that determines whether
the latent is shaped by prediction or by control.** Two consequences worth stating plainly:

- The identifiability effect we measure (actor vs observer) is *by construction* a function of that weighting. A
  larger policy weight should push the latent further toward control-relevant structure; a larger prediction weight
  toward the observer's solution. A **weight sweep is the natural next experiment**, and it is cheap.
- It also explains why the effect shrank when both models were trained longer (see the summary): with enough
  optimisation the prediction term dominates the representation for both roles.

*(This is the point of contact with the "unpredictable death" idea: a reward derived from prediction error itself
would couple the two objectives rather than summing them arbitrarily — see the discussion in
`endogenous_grabbability.ipynb`.)*

## Summary — interpretation (calibrated)

**Current results (updated 2026-07-29 — supersedes the 2026-07-28 magnitudes).**

At the original (**weak**) configuration the goal-directed actor's latent looked *much* more linearly readable than
its observer twin (position R² Δ **+0.173 / +0.135** across two seeds, velocity Δ **+0.168 / +0.165**), while the
no-goal levels 1 and 2 were nulls (Δ ≈ 0.00 / −0.01) — suggesting goal-directed agency strongly reshapes the latent.

**Training both roles to strength substantially revises this.** At 512 hidden units, MLP encoder/decoder, a 5-step
free-run loss and 25 000 iterations: the **position gap essentially disappears** (Δ **+0.030 / +0.005**) — and at
that size it is *indistinguishable from the strong no-goal control* (`L2s0`, Δ +0.018), i.e. **no longer
goal-specific at all**; the observer simply catches up (0.589 → 0.863). What **survives** at strength is (i) a
smaller but goal-specific **velocity** advantage (Δ **+0.044 / +0.060**, control −0.015) and (ii) a **canonicality**
advantage that *flips sign* to favour the actor (fiber MLP Δ **−0.070 / −0.084**, control −0.026).

**Reading (interpretation).** Goal-directed endogenous action appears mainly to **accelerate** the emergence of
linearly-readable structure rather than to produce a large, durable representational advantage; what durably
survives is a modest gain in velocity readability and canonicality. The strong-capacity **no-goal control remains a
null**, so what remains *is* attributable to the goal rather than to capacity or to action-generation per se.

**Caveats.** GRU only; god's-hand (not embodied); 2 objects; single eval trace per run; 2 seeds at level 3 and one
seed elsewhere. The prediction-vs-policy loss weighting was **never swept** (see the section above), and the
identifiability effect is by construction sensitive to it. Editability is treated in
`endogenous_grabbability.ipynb`; qualitative behaviour in `endogenous_agent_animations.ipynb`.